# Preparación reutilizable de conformeros 3D

Este notebook genera y persiste un único conformero 3D por molécula para cualquier benchmark compatible con TDC ADMET.

Para cambiar de dataset, modifica únicamente DATASET_NAME en la celda de configuración. Las salidas se guardan en data/conformers/<dataset_slug>/.

## Política de generación

La selección reproduce la estrategia de prepare_3d_molecule() de 03_feature_engineering:

1. Generar varios conformeros mediante ETKDGv3 con hidrógenos explícitos.
2. Si MMFF tiene parámetros para toda la molécula, optimizar con MMFF y seleccionar el conformero convergido de menor energía finita.
3. Si MMFF no tiene parámetros, comprobar UFF y, si está disponible, optimizar con UFF.
4. Si el campo de fuerza aplicable no converge ningún conformero con energía finita, utilizar el primer conformero ETKDG.
5. Si no hay parámetros MMFF ni UFF, utilizar directamente el primer conformero ETKDG.
6. Validar la geometría y persistir exactamente el conformero seleccionado.

**Se conserva un detalle importante de 03:** si MMFF tiene parámetros pero no converge, se utiliza ETKDG directamente; no se intenta UFF. UFF solo se intenta cuando MMFF no dispone de parámetros.

La función get_lowest_energy_conformer() devuelve el RDKit conf_id real. Para recuperar la energía asociada se localiza su posición con conf_ids.index(best_conf_id).

## Artefactos

- conformers.pkl: registros válidos, con una única conformación RDKit cada uno.
- manifest.json: configuración, versiones, cobertura y métodos utilizados.
- excluded.csv: registro de cada molécula excluida y la causa.

Los conformeros ETKDG de fallback quedan identificados explícitamente; no se presentan como geometrías optimizadas.

In [ ]:
import hashlib
import json
import pickle
import platform
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, rdBase
from rdkit.Chem import AllChem
from tdc.benchmark_group import admet_group
from tqdm.auto import tqdm

In [ ]:
# ============================================================
# Dataset: change this variable to prepare another TDC benchmark
# ============================================================

DATASET_NAME = "Solubility_AqSolDB"

# Reproducible conformer-generation settings
BASE_SEED = 42
N_CONFORMERS = 5
PRUNE_RMS_THRESH = 0.5
NUM_THREADS = 1
MAX_ITERS = 200

# Reject only severely degenerate geometries. Distances are in Angstrom.
MIN_PAIRWISE_DISTANCE_ANGSTROM = 0.05

if not isinstance(DATASET_NAME, str) or not DATASET_NAME.strip():
    raise ValueError("DATASET_NAME must be a non-empty TDC benchmark name.")

if N_CONFORMERS < 1:
    raise ValueError("N_CONFORMERS must be at least 1.")

if NUM_THREADS < 1 or MAX_ITERS < 1:
    raise ValueError("NUM_THREADS and MAX_ITERS must be positive.")


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "features" / "conformers.py").is_file():
            return candidate
    raise RuntimeError(
        "Could not locate the repository root. Run this notebook "
        "from the repository or one of its subdirectories."
    )


def dataset_slug(dataset_name):
    slug = re.sub(r"[^a-z0-9]+", "_", dataset_name.lower()).strip("_")

    # Preserve the existing G7 artifact path for Solubility_AqSolDB:
    # data/conformers/aqsoldb/conformers.pkl
    if slug.startswith("solubility_"):
        slug = slug[len("solubility_"):]

    if not slug:
        raise ValueError(f"Could not derive an output directory from {dataset_name!r}.")

    return slug


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
DATASET_SLUG = dataset_slug(DATASET_NAME)
OUTPUT_DIR = REPO_ROOT / "data" / "conformers" / DATASET_SLUG

from src.features.conformers import (
    generate_conformers,
    optimize_conformers,
    get_lowest_energy_conformer,
    keep_only_conformer,
    validate_conformer_geometry,
)

print("Dataset:", DATASET_NAME)
print("Repository:", REPO_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Output:", OUTPUT_DIR)
print({
    "base_seed": BASE_SEED,
    "n_conformers": N_CONFORMERS,
    "prune_rms_thresh": PRUNE_RMS_THRESH,
    "num_threads": NUM_THREADS,
    "max_iters": MAX_ITERS,
    "min_pairwise_distance_angstrom": MIN_PAIRWISE_DISTANCE_ANGSTROM,
})

## 1. Cargar las particiones oficiales

Se cargan train_val y test sin alterar el orden ni reasignar moléculas. El notebook espera el esquema estándar de TDC ADMET: Drug_ID, Drug y Y.

In [ ]:
if not RAW_DATA_DIR.is_dir():
    raise FileNotFoundError(f"TDC raw-data directory not found: {RAW_DATA_DIR}")

group = admet_group(path=str(RAW_DATA_DIR))
benchmark = group.get(DATASET_NAME)

required_splits = ("train_val", "test")
missing_splits = [name for name in required_splits if name not in benchmark]
if missing_splits:
    raise ValueError(
        f"{DATASET_NAME} is missing required TDC partitions: {missing_splits}"
    )

required_columns = {"Drug_ID", "Drug", "Y"}
partition_frames = []
partition_sizes = {}

for split_name in required_splits:
    frame = benchmark[split_name].copy()
    missing_columns = required_columns - set(frame.columns)

    if missing_columns:
        raise ValueError(
            f"{DATASET_NAME}/{split_name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    partition_sizes[split_name] = int(len(frame))
    partition_frames.append(frame.assign(split=split_name))

benchmark_records = (
    pd.concat(partition_frames, ignore_index=True)
    .rename(columns={
        "Drug_ID": "mol_id",
        "Drug": "smiles",
        "Y": "y",
    })
)

if benchmark_records.empty:
    raise ValueError(f"{DATASET_NAME} contains no records.")

print("Dataset:", DATASET_NAME)
print("Partition sizes:", partition_sizes)
print("Total input records:", len(benchmark_records))
display(benchmark_records.head())

## 2. Generación y selección del conformero

La función siguiente implementa la política de 03_feature_engineering, adaptada al contrato de persistencia G7.

- Solo se consideran convergidos los resultados con status 0 y energía finita.
- get_lowest_energy_conformer(conf_ids, results) devuelve el identificador real de RDKit.
- En caso de fallback se utiliza conf_ids[0], que corresponde al primer conformero ETKDG generado.
- Los errores de generación u optimización se devuelven con su etapa para poder auditarlos.

In [ ]:
def seed_for_molecule(mol_id, split, row_index):
    """Stable per-molecule seed, with a row-index fallback for missing IDs."""
    if pd.notna(mol_id):
        identity = f"id:{mol_id}"
    else:
        identity = f"{split}:row:{row_index}"

    seed_material = f"{BASE_SEED}:{DATASET_NAME}:{identity}"
    digest = hashlib.sha256(seed_material.encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "big") % (2**31 - 1) + 1


def select_lowest_converged_conformer(conf_ids, optimization_results):
    """
    Return (real RDKit conf_id, energy) for the lowest-energy converged
    conformer. Return None if no status-0 result has a finite energy.
    """
    if len(conf_ids) != len(optimization_results):
        raise ValueError(
            "Conformer IDs and optimization results are misaligned."
        )

    eligible = []
    for position, result in enumerate(optimization_results):
        try:
            status, energy = result
            energy = float(energy)
        except (TypeError, ValueError, OverflowError):
            continue

        if int(status) == 0 and np.isfinite(energy):
            eligible.append((position, energy))

    if not eligible:
        return None

    # This helper now returns the actual RDKit conformer ID, not a position.
    best_conf_id = get_lowest_energy_conformer(
        conf_ids,
        optimization_results,
    )

    # Recover the corresponding result position only to retrieve its energy.
    best_position = conf_ids.index(best_conf_id)
    best_energy = float(optimization_results[best_position][1])

    if not np.isfinite(best_energy):
        raise RuntimeError("Selected conformer has a non-finite energy.")

    return int(best_conf_id), best_energy


def prepare_3d_molecule(source_mol, random_seed):
    """
    Generate conformers and select MMFF, UFF, or ETKDG fallback.

    Returns a dictionary containing the molecule, selected RDKit conf_id,
    method metadata, or a structured failure.
    """
    stage = "conformer_embedding"

    try:
        mol_3d, conf_ids = generate_conformers(
            source_mol,
            n_conformers=N_CONFORMERS,
            random_seed=random_seed,
            prune_rms_thresh=PRUNE_RMS_THRESH,
            num_threads=NUM_THREADS,
        )

        stage = "force_field_optimization"

        # 1. Prefer MMFF when parameters are available.
        if AllChem.MMFFHasAllMoleculeParams(mol_3d):
            attempted_force_field = "MMFF"

            results = optimize_conformers(
                mol_3d,
                force_field="MMFF",
                max_iters=MAX_ITERS,
                num_threads=NUM_THREADS,
            )

            selected = select_lowest_converged_conformer(
                conf_ids,
                results,
            )

            if selected is not None:
                conf_id, energy = selected
                method = "MMFF"
                force_field = "MMFF"
                fallback_reason = None
            else:
                # Match 03_feature_engineering: do not try UFF here.
                conf_id = int(conf_ids[0])
                energy = None
                method = "ETKDG_fallback"
                force_field = None
                fallback_reason = "MMFF produced no converged conformer with finite energy."

        # 2. Try UFF only when MMFF parameters are unavailable.
        elif AllChem.UFFHasAllMoleculeParams(mol_3d):
            attempted_force_field = "UFF"

            results = optimize_conformers(
                mol_3d,
                force_field="UFF",
                max_iters=MAX_ITERS,
                num_threads=NUM_THREADS,
            )

            selected = select_lowest_converged_conformer(
                conf_ids,
                results,
            )

            if selected is not None:
                conf_id, energy = selected
                method = "UFF"
                force_field = "UFF"
                fallback_reason = None
            else:
                conf_id = int(conf_ids[0])
                energy = None
                method = "ETKDG_fallback"
                force_field = None
                fallback_reason = "UFF produced no converged conformer with finite energy."

        # 3. No force-field parameters are available.
        else:
            attempted_force_field = None
            conf_id = int(conf_ids[0])
            energy = None
            method = "ETKDG_fallback"
            force_field = None
            fallback_reason = "Neither MMFF nor UFF parameters are available."

        if int(conf_id) not in [int(value) for value in conf_ids]:
            raise RuntimeError("Selected conf_id is not present in the generated molecule.")

        return {
            "mol": mol_3d,
            "conf_id": int(conf_id),
            "conformer_method": method,
            "force_field": force_field,
            "force_field_attempted": attempted_force_field,
            "energy": energy,
            "fallback_reason": fallback_reason,
            "failure": None,
        }

    except (ValueError, RuntimeError) as exc:
        return {
            "mol": None,
            "conf_id": None,
            "conformer_method": None,
            "force_field": None,
            "force_field_attempted": None,
            "energy": None,
            "fallback_reason": None,
            "failure": {
                "stage": stage,
                "error_type": type(exc).__name__,
                "reason": str(exc),
            },
        }

## 3. Preparar cada registro

Cada molécula se procesa de forma independiente. Los fallos no interrumpen el dataset completo: se registran en excluded.csv con su etapa y motivo.

La validación geométrica se aplica también a los conformeros ETKDG de fallback. Una geometría inválida no se persiste.

In [ ]:
def prepare_record(row, row_index):
    mol_id = row.get("mol_id")
    smiles = row.get("smiles")
    target = row.get("y")
    split = row.get("split")

    seed = seed_for_molecule(mol_id, split, row_index)
    stage = "input_validation"

    try:
        if pd.isna(mol_id):
            raise ValueError("mol_id is missing.")

        if pd.isna(smiles) or not str(smiles).strip():
            raise ValueError("SMILES is missing.")

        try:
            target = float(target)
        except (TypeError, ValueError, OverflowError) as exc:
            raise ValueError("Target y is missing or non-numeric.") from exc

        if not np.isfinite(target):
            raise ValueError("Target y is non-finite.")

        if split not in {"train_val", "test"}:
            raise ValueError(f"Unexpected split: {split!r}.")

        stage = "smiles_parsing"
        source_mol = Chem.MolFromSmiles(str(smiles))

        if source_mol is None:
            raise ValueError("RDKit MolFromSmiles returned None.")

        stage = "conformer_preparation"
        prepared = prepare_3d_molecule(
            source_mol,
            random_seed=seed,
        )

        if prepared["failure"] is not None:
            failure = prepared["failure"]
            return None, {
                "input_row_index": int(row_index),
                "mol_id": mol_id,
                "smiles": smiles,
                "y": target,
                "split": split,
                "seed": int(seed),
                "stage": failure["stage"],
                "error_type": failure["error_type"],
                "reason": failure["reason"],
            }

        mol_3d = prepared["mol"]
        conf_id = int(prepared["conf_id"])

        stage = "geometry_validation"
        validate_conformer_geometry(
            mol_3d,
            conf_id,
            min_pairwise_distance=MIN_PAIRWISE_DISTANCE_ANGSTROM,
        )

        stage = "selected_conformer_persistence"
        selected_mol = keep_only_conformer(
            mol_3d,
            conf_id,
        )

        if selected_mol.GetNumConformers() != 1:
            raise RuntimeError("Expected exactly one persisted conformer.")

        persisted_conf_id = int(selected_mol.GetConformer().GetId())
        if persisted_conf_id != conf_id:
            raise RuntimeError(
                "The selected RDKit conf_id was not preserved during persistence."
            )

        num_explicit_hydrogens = sum(
            atom.GetAtomicNum() == 1
            for atom in selected_mol.GetAtoms()
        )
        num_heavy_atoms = sum(
            atom.GetAtomicNum() > 1
            for atom in selected_mol.GetAtoms()
        )

        record = {
            "input_row_index": int(row_index),
            "mol_id": mol_id,
            "smiles": str(smiles),
            "y": float(target),
            "split": str(split),
            "seed": int(seed),
            "mol": selected_mol,
            "conf_id": int(conf_id),
            "conformer_method": prepared["conformer_method"],
            "force_field": prepared["force_field"],
            "force_field_attempted": prepared["force_field_attempted"],
            "energy": prepared["energy"],
            "fallback_reason": prepared["fallback_reason"],
            "num_atoms": int(selected_mol.GetNumAtoms()),
            "num_heavy_atoms": int(num_heavy_atoms),
            "num_explicit_hydrogens": int(num_explicit_hydrogens),
        }

        return record, None

    except Exception as exc:
        return None, {
            "input_row_index": int(row_index),
            "mol_id": mol_id,
            "smiles": smiles,
            "y": target,
            "split": split,
            "seed": int(seed),
            "stage": stage,
            "error_type": type(exc).__name__,
            "reason": str(exc),
        }


records = []
excluded = []

rows = benchmark_records.to_dict(orient="records")

for row_index, row in enumerate(
    tqdm(rows, total=len(rows), desc=f"{DATASET_NAME} conformers")
):
    record, failure = prepare_record(row, row_index)

    if record is not None:
        records.append(record)
    else:
        excluded.append(failure)

if len(records) + len(excluded) != len(benchmark_records):
    raise RuntimeError(
        "Some input rows were neither persisted nor recorded as excluded."
    )

if not records:
    raise RuntimeError(
        "No valid conformers were generated; inspect the exclusions."
    )

print("Input records:", len(benchmark_records))
print("Valid records:", len(records))
print("Excluded records:", len(excluded))

In [ ]:
# Coverage and fallback diagnostics

valid_by_split = Counter(record["split"] for record in records)
excluded_by_split = Counter(item["split"] for item in excluded)
method_counts = Counter(record["conformer_method"] for record in records)
force_field_counts = Counter(
    record["force_field"] if record["force_field"] is not None else "ETKDG_fallback"
    for record in records
)
fallback_reason_counts = Counter(
    record["fallback_reason"]
    for record in records
    if record["fallback_reason"] is not None
)
excluded_by_stage = Counter(item["stage"] for item in excluded)
excluded_by_error_type = Counter(item["error_type"] for item in excluded)

summary = pd.DataFrame([
    {
        "split": split_name,
        "input": int(partition_sizes[split_name]),
        "valid": int(valid_by_split[split_name]),
        "excluded": int(excluded_by_split[split_name]),
        "coverage": (
            valid_by_split[split_name] / partition_sizes[split_name]
            if partition_sizes[split_name] else np.nan
        ),
    }
    for split_name in required_splits
])

display(summary)
display(
    pd.DataFrame(
        [{"conformer_method": key, "count": int(value)}
         for key, value in method_counts.items()]
    ).sort_values("count", ascending=False)
)

if fallback_reason_counts:
    display(
        pd.DataFrame(
            [{"fallback_reason": key, "count": int(value)}
             for key, value in fallback_reason_counts.items()]
        ).sort_values("count", ascending=False)
    )

if excluded:
    display(
        pd.DataFrame(
            [{"stage": key, "count": int(value)}
             for key, value in excluded_by_stage.items()]
        ).sort_values("count", ascending=False)
    )
    display(pd.DataFrame(excluded).head(20))

## 4. Crear el manifest y persistir los artefactos

El manifest registra tanto los parámetros globales como el método realmente utilizado para cada molécula. Los registros ETKDG de fallback tienen energy=None y force_field=None; no se confunden con conformeros optimizados.

In [ ]:
manifest = {
    "format_version": 2,
    "dataset": DATASET_NAME,
    "dataset_slug": DATASET_SLUG,
    "source": "TDC admet_group official train_val and test partitions",
    "input_records": int(len(benchmark_records)),
    "valid_records": int(len(records)),
    "excluded_records": int(len(excluded)),
    "input_records_by_split": {
        str(key): int(value)
        for key, value in benchmark_records["split"].value_counts().items()
    },
    "valid_records_by_split": {
        str(key): int(value)
        for key, value in valid_by_split.items()
    },
    "excluded_records_by_split": {
        str(key): int(value)
        for key, value in excluded_by_split.items()
    },
    "conformer_method_counts": {
        str(key): int(value)
        for key, value in method_counts.items()
    },
    "force_field_counts": {
        str(key): int(value)
        for key, value in force_field_counts.items()
    },
    "fallback_reason_counts": {
        str(key): int(value)
        for key, value in fallback_reason_counts.items()
    },
    "excluded_records_by_stage": {
        str(key): int(value)
        for key, value in excluded_by_stage.items()
    },
    "excluded_records_by_error_type": {
        str(key): int(value)
        for key, value in excluded_by_error_type.items()
    },
    "seed": {
        "base_seed": int(BASE_SEED),
        "strategy": "SHA256(base_seed + dataset_name + stable mol_id; split/row index fallback if ID missing)",
        "per_record_field": "seed",
    },
    "embedding": {
        "method": "ETKDGv3",
        "n_conformers": int(N_CONFORMERS),
        "prune_rms_thresh_angstrom": float(PRUNE_RMS_THRESH),
        "num_threads": int(NUM_THREADS),
    },
    "optimization": {
        "max_iters": int(MAX_ITERS),
        "num_threads": int(NUM_THREADS),
        "eligible_status": 0,
        "eligible_energy": "finite",
        "policy": [
            "Use MMFF when all molecule parameters are available.",
            "If MMFF parameters are unavailable, try UFF when all UFF parameters are available.",
            "If the selected force field produces no converged finite-energy conformer, use the first ETKDG conformer.",
            "If neither force field has parameters, use the first ETKDG conformer.",
            "When MMFF parameters exist but MMFF does not converge, do not try UFF; this matches 03_feature_engineering.",
        ],
    },
    "geometry_validation": {
        "coordinates_finite": True,
        "minimum_pairwise_distance_angstrom": float(
            MIN_PAIRWISE_DISTANCE_ANGSTROM
        ),
        "planar_geometries_allowed": True,
    },
    "hydrogens": {
        "policy": "explicit",
        "added_with": "Chem.AddHs",
        "removed": False,
        "note": "Some valid molecules legitimately contain zero hydrogen atoms; hydrogen presence is not required.",
    },
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "rdkit": rdBase.rdkitVersion,
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

conformers_path = OUTPUT_DIR / "conformers.pkl"
manifest_path = OUTPUT_DIR / "manifest.json"
excluded_path = OUTPUT_DIR / "excluded.csv"

with conformers_path.open("wb") as handle:
    pickle.dump(records, handle, protocol=pickle.HIGHEST_PROTOCOL)

manifest_path.write_text(
    json.dumps(manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

excluded_columns = [
    "input_row_index",
    "mol_id",
    "smiles",
    "y",
    "split",
    "seed",
    "stage",
    "error_type",
    "reason",
]
pd.DataFrame(excluded, columns=excluded_columns).to_csv(
    excluded_path,
    index=False,
)

print("Saved:", conformers_path)
print("Saved:", manifest_path)
print("Saved:", excluded_path)

## 5. Verificación de persistencia

Se vuelve a abrir el pickle y se comprueba el contrato que consume 07_gnn_graph_generation.ipynb:

- cada registro contiene los campos mínimos;
- cada molécula conserva exactamente un conformero;
- el conf_id real de RDKit se conserva;
- las coordenadas son finitas y la geometría pasa la validación;
- el número de hidrógenos explícitos coincide con los átomos H presentes;
- los fallbacks ETKDG están etiquetados y no tienen energía de campo de fuerza.

No se exige que todas las moléculas contengan al menos un átomo de hidrógeno: hay moléculas químicamente válidas sin hidrógenos.

In [ ]:
with conformers_path.open("rb") as handle:
    persisted = pickle.load(handle)

required_fields = {
    "mol_id",
    "smiles",
    "y",
    "split",
    "seed",
    "mol",
    "conf_id",
}

assert isinstance(persisted, list)
assert len(persisted) == len(records)
assert all(required_fields <= set(record) for record in persisted)
assert all(record["mol"].GetNumConformers() == 1 for record in persisted)

for record in persisted:
    mol = record["mol"]
    conf_id = int(record["conf_id"])

    assert int(mol.GetConformer().GetId()) == conf_id
    assert int(mol.GetNumAtoms()) == int(record["num_atoms"])

    actual_hydrogens = sum(
        atom.GetAtomicNum() == 1
        for atom in mol.GetAtoms()
    )
    assert actual_hydrogens == int(record["num_explicit_hydrogens"])

    validate_conformer_geometry(
        mol,
        conf_id,
        min_pairwise_distance=MIN_PAIRWISE_DISTANCE_ANGSTROM,
    )

    if record["conformer_method"] in {"MMFF", "UFF"}:
        assert record["force_field"] == record["conformer_method"]
        assert record["energy"] is not None
        assert np.isfinite(float(record["energy"]))
        assert record["fallback_reason"] is None

    elif record["conformer_method"] == "ETKDG_fallback":
        assert record["force_field"] is None
        assert record["energy"] is None
        assert record["fallback_reason"] is not None

    else:
        raise AssertionError(
            f"Unknown conformer method: {record['conformer_method']!r}"
        )

assert len(persisted) + len(excluded) == len(benchmark_records)
assert manifest["valid_records"] == len(persisted)
assert manifest["excluded_records"] == len(excluded)

print("Persistence checks passed.")
print("Dataset:", DATASET_NAME)
print("Records ready for graph generation:", len(persisted))
print("Excluded records:", len(excluded))
print("Conformer methods:", dict(method_counts))
print("Output directory:", OUTPUT_DIR)